# CSAW26-QFLARE
## Demo (introduction and walkthrough)

This notebook introduces the model, explains where your attack enters the training process, and runs the same evaluation used by the command-line tools.

**Your task** is to reduce a shared classifier's test macro-F1 by changing the circuit updates sent by two of eight clients. You may change updates in at most **30 of the 60 rounds** in each evaluation case.

### Before running

1. Extract or clone the student repository.
2. In that repository's Python environment, run `python -m pip install -r requirements-notebook.txt`.
3. Open this notebook from the repository folder in Jupyter or VS Code and select that Python environment as the kernel.
4. If you installed dependencies in an already running kernel, restart it once.
5. Select **Run All**.

The complete evaluation can take several minutes on a CPU. It writes a fresh `outputs/result.json` for the current `attack.py`. Start with the provided attack, then edit `attack.py` and rerun the evaluation cells.

Read [README.md](README.md) for the rules and [hints.md](hints.md) (it will be uploaded by Friday 2pm GST time) for development guidance.

## Background: what is quantum federated learning?

### Federated learning

In federated learning, several clients train a shared model using their own local datasets. A server sends the current model parameters to the clients. Each client trains locally and sends its updated parameters back. The server combines these updates to form the next shared model. This cycle is called a **training round**.

Clients exchange model parameters without sending their raw training examples to the server. This alone does not guarantee privacy, because updates can still contain information about local data. In this challenge, Flower handles the client interface and aggregation, with all clients simulated locally on one computer.

### The quantum part

Each client trains a **hybrid quantum-classical classifier**. Four image features are encoded as qubit rotations. A trainable quantum circuit then applies rotation gates and entangling gates. Its rotation angles are learned during training, much like weights in a neural network.

The circuit produces four measurement expectation values. A classical linear layer converts these values into scores for digits 3 and 8. Both the circuit angles and the classical layer are trained from the classification loss.

Here, PennyLane simulates the circuit using a full statevector. With `shots=None`, measurement expectations are calculated analytically, subject to floating-point precision, without sampling a finite number of measurement shots. The clients exchange **classical parameter arrays**; the training setup does not require quantum communication.

### Why can malicious updates cause damage?

Federated training depends on the updates clients send. A compromised client can send an update designed to move the shared model away from a good classifier. This is a **model-poisoning attack**.

In this challenge, your function replaces the quantum circuit updates from two clients. The remaining six clients train honestly. Your attack is restricted by a perturbation bound and a budget of 30 changed rounds. These limits make the choice of direction, coordination, and attack timing part of the task.

### Why test three aggregation methods?

| Method | How it combines client models | Why it matters for the attack |
| --- | --- | --- |
| FedAvg | Takes an average weighted by client sample counts. | Malicious values can influence the average. |
| Median | Takes the median separately for each parameter coordinate. | Extreme values have less direct influence, but harmful updates may still affect the result. |
| Trimmed mean | Removes the lowest and highest values for each coordinate, then averages the remainder. | The attack must account for the possibility that extreme values are discarded. |

These methods do not guarantee protection against every attack. The benchmark measures what happens under its specific model, data split, and attack limits.

For each evaluation case, clean and attacked training start from the same conditions. **Macro-F1** averages the F1 scores of the two classes equally. Your score measures how much your attack reduces this metric, averaged across the six evaluation cases. An attack can also fail or improve performance, so the saved results are needed to assess its effect.


In [ ]:
from pathlib import Path
import sys
import json

ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "rules.json").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted student repository.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pennylane as qml
import flwr
from IPython.display import display, HTML

from score import RULES
if RULES["benchmark"] != "qfl-flower-pennylane-digits38-v4":
    raise RuntimeError("Use the current student repository and restart the notebook kernel.")

print("Python", sys.version.split()[0])
print("PennyLane", qml.__version__)
print("Flower", flwr.__version__)
print("Repository", ROOT)

## 1. What model are the clients training?

The task is handwritten digit classification using digits **3** and **8** from scikit-learn Digits. Internally, digit 3 has label 0 and digit 8 has label 1.

| Stage | What happens |
| --- | --- |
| Image preprocessing | A 64-pixel image is reduced to four PCA features using a transform fitted on training data. |
| Data encoding | `AngleEmbedding` encodes the four features as Y rotations on four qubits. |
| Trainable quantum circuit | Two `StronglyEntanglingLayers` contain 24 trainable rotation angles. |
| Measurement | The circuit returns four Pauli-X expectation values. |
| Classical head | A linear layer maps these four values to two class logits, using eight weights and two biases. |

Each local model has **34 trainable parameters**. Clients train the circuit and head jointly using SGD and PennyLane NumPy/Autograd.

The quantum device is `default.qubit` with `shots=None`. It evolves the full statevector and evaluates exact expectations in floating-point arithmetic. PCA is explicit input dimensionality reduction.

The next cell runs one **untrained forward pass** to show the input and output shapes. Its prediction illustrates the model interface.

In [ ]:
from challenge import prepare
from quantum_model import forward

example_data = prepare(RULES["public_seeds"][0])
features = example_data["features"][:1]
initial_params = example_data["initial"]
logits = np.asarray(forward(features, initial_params))

probabilities = np.exp(logits - logits.max(axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
predicted_label = int(probabilities.argmax(axis=1)[0])

print("One input sample", features.shape)
print("Model parameters", initial_params.shape)
print("Two output logits", logits.shape)
print("Class probabilities", {"digit 3": float(probabilities[0, 0]),
                             "digit 8": float(probabilities[0, 1])})
print("Untrained prediction", 3 if predicted_label == 0 else 8)

## 2. Where does your attack enter?

Every training round follows these steps.

1. The server sends the current model parameters to all eight clients.
2. Each client trains on its own data and returns an updated model through Flower.
3. The evaluator gives your `attack` function the two compromised clients' honest updates.
4. Your function returns replacement **circuit deltas** with shape `(2, 24)`. The evaluator applies the perturbation and round limits.
5. Flower aggregates the eight client models using the selected defense.

A delta is the change from the current global parameters. Your output specifies deltas for the circuit's 24 angles. The evaluator uses the honest head updates from your two clients.

| Input to your function | Meaning |
| --- | --- |
| `global_params`, shape `(34,)` | Current shared parameters. |
| `own_deltas`, shape `(2, 34)` | Honest updates from your two clients. |
| `round_idx` | Current round, 0 through 59. |
| `state` | Your previous state, initially `None`. |

To skip a round, return `own_deltas[:, :24].copy(), state`. A changed round counts once whether you modify one client or both. The budget resets for each seed and defense.

The starter reverses the circuit deltas in rounds **0, 2, ..., 58** and skips the others. The next cell checks its callable interface using one illustrative update array. These values are a shape example; the scored evaluation below uses real client training.

In [ ]:
from evaluate import load_attack

attack = load_attack(ROOT / "attack.py")
example_updates = np.full((2, 34), 0.01, dtype=float)
replacement, example_state = attack(initial_params.copy(), example_updates.copy(), 0, None)
replacement = np.asarray(replacement)

print("Example input updates", example_updates.shape)
print("Returned circuit deltas", replacement.shape)
print("Finite values", bool(np.isfinite(replacement).all()))
assert replacement.shape == (2, 24)
assert replacement.dtype.kind in "fiu"
assert np.isfinite(replacement).all()

## 3. Run the complete public evaluation

Each submission is evaluated with **two seeds × three defenses = six cases**. For each case, the evaluator trains a clean model and an attacked model using the same split, initialization, client partition, and minibatch schedule.

The defenses are Flower's **FedAvg**, **FedMedian**, and **FedTrimmedAvg**. Every model trains for all 60 rounds.

The cell below runs the evaluator for your current `attack.py`. The equivalent terminal command is

```bash
python evaluate.py --submission attack.py --output outputs/result.json
```

Keep this notebook open while training completes. The saved result is written after all six cases finish.

In [ ]:
from evaluate import evaluate

RESULT_PATH = ROOT / "outputs" / "result.json"
report = evaluate(ROOT / "attack.py", RESULT_PATH)
print("Saved", RESULT_PATH)
print("Score", report["summary"]["score"], "percentage points")

## 4. Read your score

The score is the average F1 drop across all six cases, expressed in percentage points.

```text
Score = 100 * average(Clean F1 - Attacked F1)
```

Higher is better. A negative drop means the attacked model performed better in that case and reduces the mean score.

The provided starter's reference score is **0.56530779 percentage points**. Your edited attack may produce another score. Inspect every case, including those where your attack was ineffective.

In [ ]:
from statistics import mean
from IPython.display import HTML, display

runs = report["summary"]["runs"]
defenses = list(dict.fromkeys(row["defense"] for row in runs))

rows = ""
for defense in defenses:
    cases = [row for row in runs if row["defense"] == defense]
    rows += (
        f"<tr><td>{defense}</td>"
        f"<td>{mean(r['clean_f1'] for r in cases):.4f}</td>"
        f"<td>{mean(r['attacked_f1'] for r in cases):.4f}</td>"
        f"<td>{mean(r['drop_pp'] for r in cases):.4f}</td>"
        "</tr>"
    )

average_drop = mean(row["drop_pp"] for row in runs)

rows += (
    "<tr><th>All cases</th>"
    f"<th>{mean(r['clean_f1'] for r in runs):.4f}</th>"
    f"<th>{mean(r['attacked_f1'] for r in runs):.4f}</th>"
    f"<th>{average_drop:.4f}</th></tr>"
)

display(HTML(
    "<table><thead><tr>"
    "<th>Defense</th><th>Mean clean F1</th>"
    "<th>Mean attacked F1</th><th>Mean drop (pp)</th>"
    "</tr></thead><tbody>" + rows + "</tbody></table>"
))

print(f"Challenge score: {report['summary']['score']:.6f} pp")
print(f"Tie-break value: {report['summary']['tie_break']:.6f} pp")

# Check that the displayed average matches the reported score.
assert abs(average_drop - report["summary"]["score"]) < 1e-9

# Check the budget separately for every evaluation case.
assert all(
    row["attack_rounds_used"] <= RULES["max_attack_rounds"]
    for row in runs
)

## 5. Replay the saved score

The result JSON contains test labels, clean and attacked predictions, used attack rounds, configuration, and code hashes.

`score.py` recalculates the score from the saved predictions. This step runs without training. The `--submission` option checks that the saved result matches the current attack source.

The next cell uses the notebook's Python environment to run this terminal command.

```bash
python score.py outputs/result.json --submission attack.py
```

After editing `attack.py`, run the evaluation cell again to generate a matching result.

In [ ]:
import subprocess

replay_command = [
    sys.executable, str(ROOT / "score.py"), str(RESULT_PATH),
    "--submission", str(ROOT / "attack.py"),
]
completed = subprocess.run(
    replay_command, cwd=ROOT, check=True, capture_output=True, text=True
)
replayed = json.loads(completed.stdout)
assert replayed == report["summary"]
print("Saved score replay passed.")
print(json.dumps({"score": replayed["score"],
                  "tie_break": replayed["tie_break"],
                  "unit": replayed["unit"]}, indent=2))

## 6. Develop and submit your method

Edit **only `attack.py` for the required challenge**. Read [hints.md](hints.md) for possible experiments (Friday 2pm GST time). To evaluate changes, rerun the evaluation and result cells or use the terminal commands in the README.

Your private submission repository must contain the exact `attack.py` used for evaluation and its generated `outputs/result.json`. Invite **@Innanov** as a collaborator.

Optional bonus work belongs under `extensions/`. The README describes the three independent extensions, each worth up to 10 points. Model or metric changes for those experiments use separate extension code.

Use the result summary and saved round indices to explain what your attack achieved. The teaching team can replay the result and rerun the submitted source to verify it.